# Diagnóstico de paredes

> Por qué wall transfiere mal a BLD-AR: wall por píxel, por estilo de pared y por plano, tiles adaptados al grosor de muro, matriz de confusión por píxel, mapas de calor de la red y límites de YOLO-seg. Los resultados se documentan en `docs/DIAGNOSTICO_PAREDES.md`.

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 sahi==0.12.8 pycocotools==2.0.11

## Imports

In [ ]:
import json
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import shapely
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from sahi import AutoDetectionModel
from sahi.postprocess.combine import GreedyNMMPostprocess
from sahi.predict import get_sliced_prediction
from sahi.prediction import ObjectPrediction
from sahi.slicing import get_slice_bboxes
from shapely.geometry import Polygon
from ultralytics import YOLO
from ultralytics.utils import ops

## Rutas y parámetros

In [ ]:
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
BLDAR = REPO / "datasets/BLD-AR"
FPC = REPO / "datasets/floor-plan-cad-yolo"
WEIGHTS = REPO / "weights/00-YOLO26m-SEG-1024/best.pt"

TILE = 1024
OVERLAP = 0.2
CONF = 0.05
PIXEL_CONF = 0.25  # confianza mínima de una máscara para las métricas por píxel
FPC_WALL_THICKNESS = 18  # px de muro en FloorPlanCAD con imgsz 1024 (docs/EVALUACION.md)
MAX_SIDE = 6000  # los planos más grandes se analizan reducidos a este lado
HEATMAP_PLANS = ["BLD_07_AR_FP_02", "BLD_01_AR_FP_02", "BLD_14_AR_FP_01", "BLD_14_AR_FP_03", "BLD_09_AR_FP_02",
                 "test-00_0366-0005", "test-00_0551-0091"]
DEVICE = "cuda:0"

assert WEIGHTS.exists(), f"No existen los pesos: {WEIGHTS}"

## Etiquetas

> Clases de `docs/MAPEO_CLASES.md`: el resto de las etiquetas de BLD-AR se ignora en las métricas de wall. El estilo de cada pared sale de la tinta en su interior, sin los bordes, como en `docs/DIAGNOSTICO_PAREDES.md`.

In [ ]:
MAPPED = {"wall", "door", "window", "sliding_door", "stair", "elevator", "railing"}
bldar = {path.stem: json.loads(path.read_text()) for path in sorted(BLDAR.glob("*.json"))}
LABELS = ["fondo"] + sorted({shape["label"] for data in bldar.values() for shape in data["shapes"]})
WALL_LABEL = LABELS.index("wall")
IGNORED = [i for i, label in enumerate(LABELS) if label != "fondo" and label not in MAPPED]


def shape_points(shape):
    """Puntos de una forma LabelMe; los círculos se aproximan con 32 lados."""
    points = np.array(shape["points"], dtype=float)
    if shape["shape_type"] == "circle":
        center, edge = points
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        points = center + np.linalg.norm(edge - center) * np.stack([np.cos(angles), np.sin(angles)], axis=1)
    return points


def bldar_labels(data, scale):
    """Índice de LABELS por píxel a escala `scale`; wall se dibuja al final para que nada la tape."""
    raster = Image.new("I", (round(data["imageWidth"] * scale), round(data["imageHeight"] * scale)), 0)
    draw = ImageDraw.Draw(raster)
    for shape in sorted(data["shapes"], key=lambda shape: shape["label"] == "wall"):
        draw.polygon([tuple(p) for p in shape_points(shape) * scale], fill=LABELS.index(shape["label"]))
    return np.asarray(raster)


def bldar_walls(data):
    return [shape_points(shape) for shape in data["shapes"] if shape["label"] == "wall"]


def fpc_walls(image_path, width, height):
    """Polígonos de wall de una imagen del test de FloorPlanCAD, en píxeles."""
    walls = []
    for line in (FPC / "labels/test" / f"{image_path.stem}.txt").read_text().splitlines():
        values = line.split()
        if values and int(values[0]) == WALL:
            walls.append(np.array(values[1:], dtype=float).reshape(-1, 2) * [width, height])
    return walls


def thickness(walls):
    """Espesor mediano de las paredes: 2 · área / perímetro."""
    polygons = [shapely.make_valid(Polygon(points)) for points in walls if len(points) >= 3]
    values = [2 * p.area / p.length for p in polygons if p.area > 0]
    return float(np.median(values)) if values else float("nan")


def polygon_box(points, shape):
    """Caja (x0, y0, x1, y1) de un polígono, recortada a la imagen."""
    x0, y0 = np.maximum(np.floor(points.min(0)).astype(int), 0)
    x1, y1 = np.minimum(np.ceil(points.max(0)).astype(int) + 1, [shape[1], shape[0]])
    return x0, y0, x1, y1


def polygon_mask(points, box):
    x0, y0, x1, y1 = box
    mask = Image.new("1", (x1 - x0, y1 - y0), 0)
    ImageDraw.Draw(mask).polygon([tuple(p) for p in points - [x0, y0]], fill=1)
    return np.asarray(mask)


def union_mask(polygons, shape):
    mask = Image.new("1", (shape[1], shape[0]), 0)
    draw = ImageDraw.Draw(mask)
    for points in polygons:
        draw.polygon([tuple(p) for p in points], fill=1)
    return np.asarray(mask)


def wall_style(gray, points):
    """Estilo de una pared según la tinta en su interior, sin los bordes."""
    polygon = shapely.make_valid(Polygon(points))
    if polygon.area <= 0:
        return "sin interior"
    pixels = []
    for part in shapely.get_parts(polygon.buffer(-max(2.0, 0.5 * polygon.area / polygon.length))):
        if part.geom_type == "Polygon" and part.area > 0:
            inner = np.array(part.exterior.coords)
            box = polygon_box(inner, gray.shape)
            if box[2] > box[0] and box[3] > box[1]:
                pixels.append(gray[box[1]:box[3], box[0]:box[2]][polygon_mask(inner, box)])
    pixels = np.concatenate(pixels) if pixels else np.empty(0)
    if pixels.size == 0:
        return "sin interior"
    dark, light_gray = (pixels < 128).mean(), ((pixels >= 128) & (pixels < 230)).mean()
    if dark > 0.6:
        return "relleno negro"
    if light_gray > 0.5:
        return "relleno gris"
    return "con trazos adentro" if dark > 0.05 else "vacía"


def recall_counts(predicted, points):
    """Píxeles de la pared predichos como wall y píxeles totales de la pared."""
    box = polygon_box(points, predicted.shape)
    if box[2] <= box[0] or box[3] <= box[1]:
        return 0, 0
    mask = polygon_mask(points, box)
    return int(predicted[box[1]:box[3], box[0]:box[2]][mask].sum()), int(mask.sum())

## Mapas por tile

> La probabilidad de wall de un píxel es la confianza de la detección por la probabilidad de su máscara antes del umbral, la mayor entre las paredes que lo cubren. La clase predicha por píxel usa las máscaras que devuelve el modelo (umbral 0.5) con confianza ≥ `PIXEL_CONF`; si se superponen, gana la de mayor confianza. El mapa de activación es EigenCAM: las activaciones de la capa proyectadas en su componente principal, calculada sobre la imagen entera para que todos los tiles usen la misma.

In [ ]:
def tile_maps(pred, proto, shape):
    """Probabilidad de wall, clase y confianza por píxel de un tile, a la resolución de entrada de la red."""
    wall = torch.zeros(shape, device=proto.device)
    confidence = torch.zeros(shape, device=proto.device)
    classes = torch.full(shape, -1, dtype=torch.int16, device=proto.device)
    c, mh, mw = proto.shape
    for chunk in pred.split(16) if len(pred) else []:
        logits = (chunk[:, 6:] @ proto.float().view(c, -1)).view(-1, mh, mw)
        # Como ops.process_mask, pero sin binarizar
        probability = ops.crop_mask(F.interpolate(logits[None], shape, mode="bilinear")[0].sigmoid(), chunk[:, :4])
        for mask, conf, cls in zip(probability, chunk[:, 4], chunk[:, 5].long()):
            if cls == WALL:
                wall = torch.maximum(wall, mask * conf)
            pixels = (mask > 0.5) & (conf >= PIXEL_CONF) & (conf > confidence)
            confidence[pixels] = conf
            classes[pixels] = cls.to(torch.int16)
    return wall.cpu().numpy(), classes.cpu().numpy(), confidence.cpu().numpy()


def to_size(array, original, size, interpolation):
    """Lleva un mapa de la entrada de la red (con letterbox) al tamaño `size` de la región original."""
    h, w = array.shape
    gain = min(h / original[0], w / original[1])
    pad_x, pad_y = (w - original[1] * gain) / 2, (h - original[0] * gain) / 2
    array = array[round(pad_y - 0.1):h - round(pad_y + 0.1), round(pad_x - 0.1):w - round(pad_x + 0.1)]
    return cv2.resize(array, (size[1], size[0]), interpolation=interpolation)

## Modelo

> Se intercepta la salida de la red en cada tile: los coeficientes de las máscaras y sus prototipos, antes del umbral, y las activaciones de la capa que entrega a la cabeza la escala más fina (stride 8).

In [ ]:
yolo = YOLO(str(WEIGHTS))
WALL = next(i for i, name in yolo.names.items() if name == "wall")
yolo.predict(np.zeros((TILE, TILE, 3), np.uint8), imgsz=TILE, device=DEVICE, verbose=False)  # crea el predictor

captured = {}
construct_result = yolo.predictor.construct_result


def capture_result(pred, img, orig_img, img_path, proto):
    captured["maps"] = tile_maps(pred, proto, img.shape[2:])
    captured["detections"] = pred[:, 5].long().bincount(minlength=len(yolo.names)).cpu().numpy()
    return construct_result(pred, img, orig_img, img_path, proto)


yolo.predictor.construct_result = capture_result
network = yolo.predictor.model.model.model  # copia de la red que usa el predictor
network[network[-1].f[0]].register_forward_hook(lambda module, inputs, output: captured.__setitem__("features", output))

## Inferencia por imagen

In [ ]:
def image_slices(image, tile):
    """Tiles de `tile` px como los de SAHI (None: la imagen entera)."""
    H, W = image.shape[:2]
    if tile is None:
        return [[0, 0, W, H]]
    return get_slice_bboxes(H, W, slice_height=tile, slice_width=tile,
                            overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP)


def activation_basis(image, tile):
    """Media y componente principal de las activaciones en toda la imagen, la base de EigenCAM."""
    total, outer, count = 0, 0, 0
    for x0, y0, x1, y1 in image_slices(image, tile):
        yolo.predict(image[y0:y1, x0:x1], imgsz=TILE, conf=CONF, device=DEVICE, verbose=False)
        x = captured["features"][0].double().flatten(1).T
        total, outer, count = total + x.sum(0), outer + x.T @ x, count + len(x)
    mean = total / count
    return mean, torch.linalg.eigh(outer / count - torch.outer(mean, mean)).eigenvectors[:, -1]


def predict_image(image, tile, scale, cam_basis=None):
    """Mapas de una imagen a escala `scale`, por tiles de `tile` px; con `cam_basis`, también EigenCAM."""
    H, W = image.shape[:2]
    size = (round(H * scale), round(W * scale))
    maps = {"wall": np.zeros(size, np.float32), "classes": np.full(size, -1, np.int16),
            "confidence": np.zeros(size, np.float32), "detections": []}
    cam, norm, count = np.zeros(size, np.float32), np.zeros(size, np.float32), np.zeros(size, np.float32)
    for x0, y0, x1, y1 in image_slices(image, tile):
        yolo.predict(image[y0:y1, x0:x1], imgsz=TILE, conf=CONF, device=DEVICE, verbose=False)
        wall, classes, confidence = captured["maps"]
        X0, Y0, X1, Y1 = (round(v * scale) for v in (x0, y0, x1, y1))
        original, region, out = (y1 - y0, x1 - x0), np.s_[Y0:Y1, X0:X1], (Y1 - Y0, X1 - X0)
        np.maximum(maps["wall"][region], to_size(wall, original, out, cv2.INTER_LINEAR), out=maps["wall"][region])
        confidence = to_size(confidence, original, out, cv2.INTER_NEAREST)
        better = confidence > maps["confidence"][region]
        maps["confidence"][region][better] = confidence[better]
        maps["classes"][region][better] = to_size(classes, original, out, cv2.INTER_NEAREST)[better]
        maps["detections"].append(captured["detections"])
        if cam_basis is not None:
            x = captured["features"][0].double()
            projection = torch.einsum("chw,c->hw", x - cam_basis[0][:, None, None], cam_basis[1])
            cam[region] += to_size(projection.float().cpu().numpy(), original, out, cv2.INTER_LINEAR)
            norm[region] += to_size(x.norm(dim=0).float().cpu().numpy(), original, out, cv2.INTER_LINEAR)
            count[region] += 1
    if cam_basis is not None:
        cam, norm = cam / np.maximum(count, 1), norm / np.maximum(count, 1)
        if np.sum((cam - cam.mean()) * (norm - norm.mean())) < 0:  # el signo de la componente es arbitrario
            cam = -cam
        cam = np.clip(cam, 0, None)
        maps["cam"] = np.clip(cam / max(np.percentile(cam, 99.5), 1e-6), 0, 1)
    return maps


def thumbnail(array, side=1600):
    """Copia reducida para los gráficos, y su escala."""
    k = min(1.0, side / max(array.shape))
    return cv2.resize(array, (round(array.shape[1] * k), round(array.shape[0] * k)), interpolation=cv2.INTER_AREA), k

## BLD-AR con tiles de 1024 px

> Por plano: wall por píxel, recall de cada pared según su estilo, matriz de confusión y detecciones por tile. Guarda los mapas de los planos de `HEATMAP_PLANS`.

In [ ]:
NO_PREDICTION = len(yolo.names)  # columna de la matriz para los píxeles sin predicción
confusion = np.zeros((len(LABELS), NO_PREDICTION + 1), np.int64)
plans, styles, tiles, heatmaps = [], [], [], {}
for name, data in bldar.items():
    image = cv2.imread(str(BLDAR / data["imagePath"]))
    scale = min(1.0, MAX_SIDE / max(image.shape[:2]))
    maps = predict_image(image, TILE, scale, activation_basis(image, TILE) if name in HEATMAP_PLANS else None)
    labels = bldar_labels(data, scale)
    predicted = np.where(maps["classes"] < 0, NO_PREDICTION, maps["classes"])
    confusion += np.bincount((labels * (NO_PREDICTION + 1) + predicted).ravel(),
                             minlength=confusion.size).reshape(confusion.shape)
    gt_wall, pred_wall, valid = labels == WALL_LABEL, predicted == WALL, ~np.isin(labels, IGNORED)
    walls = bldar_walls(data)
    plans.append({"plano": name, "espesor": thickness(walls), "tp": (gt_wall & pred_wall).sum(),
                  "fp": (~gt_wall & pred_wall & valid).sum(), "fn": (gt_wall & ~pred_wall).sum()})
    gray = cv2.resize(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), labels.shape[::-1], interpolation=cv2.INTER_AREA)
    styles += [("BLD-AR", name, wall_style(gray, points * scale), *recall_counts(pred_wall, points * scale))
               for points in walls]
    tiles += maps["detections"]
    if name in HEATMAP_PLANS:
        small, k = thumbnail(gray)
        heatmaps[name] = (small, [points * scale * k for points in walls], thumbnail(maps["wall"])[0],
                          thumbnail(maps["cam"])[0])
print(f"{len(plans)} planos, {len(tiles)} tiles")

## FloorPlanCAD: test completo

> Una inferencia por imagen, sin tiles, como en la evaluación de test del entrenamiento.

In [ ]:
fpc_images = sorted((FPC / "images/test").iterdir())
fpc_totals = np.zeros(3, np.int64)
for path in fpc_images:
    image = cv2.imread(str(path))
    H, W = image.shape[:2]
    maps = predict_image(image, None, 1.0, activation_basis(image, None) if path.stem in HEATMAP_PLANS else None)
    walls = fpc_walls(path, W, H)
    gt_wall, pred_wall = union_mask(walls, (H, W)), maps["classes"] == WALL
    fpc_totals += [(gt_wall & pred_wall).sum(), (~gt_wall & pred_wall).sum(), (gt_wall & ~pred_wall).sum()]
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    styles += [("FloorPlanCAD", path.stem, wall_style(gray, points), *recall_counts(pred_wall, points))
               for points in walls]
    if path.stem in HEATMAP_PLANS:
        small, k = thumbnail(gray)
        heatmaps[path.stem] = (small, [points * k for points in walls], thumbnail(maps["wall"])[0],
                               thumbnail(maps["cam"])[0])
print(f"{len(fpc_images)} imágenes")

## Wall por píxel

> Sin instancias: separa "no encuentra las paredes" de "las encuentra pero las divide distinto que la etiqueta".

In [ ]:
def show_table(header, rows):
    cell = lambda v: f"{v:.3f}" if isinstance(v, float) else str(v)
    display(Markdown("| " + " | ".join(header) + " |\n|" + "---|" * len(header) + "\n"
                     + "\n".join("| " + " | ".join(cell(v) for v in row) + " |" for row in rows)))


def pixel_scores(tp, fp, fn):
    """IoU, precisión, recall y F1."""
    tp, fp, fn = float(tp), float(fp), float(fn)
    return [tp / max(tp + fp + fn, 1), tp / max(tp + fp, 1), tp / max(tp + fn, 1), 2 * tp / max(2 * tp + fp + fn, 1)]


bldar_totals = np.sum([[plan["tp"], plan["fp"], plan["fn"]] for plan in plans], axis=0)
show_table(["Wall por píxel", "IoU", "Precisión", "Recall", "F1"],
           [["FloorPlanCAD (test)", *pixel_scores(*fpc_totals)], ["BLD-AR", *pixel_scores(*bldar_totals)]])

## Wall por estilo de pared

> Recall por píxel de las paredes etiquetadas, según la tinta en su interior: si el modelo depende de los trazos dentro de la pared, las vacías tienen menos recall también en FloorPlanCAD.

In [ ]:
by_style = {}
for dataset, _, style, hits, total in styles:
    entry = by_style.setdefault(style, {}).setdefault(dataset, [0, 0, 0])
    entry[0], entry[1], entry[2] = entry[0] + 1, entry[1] + hits, entry[2] + total


def style_cells(entry):
    return [entry[0], entry[1] / entry[2] if entry[2] else "-"] if entry else [0, "-"]


show_table(["Estilo", "FloorPlanCAD: paredes", "FloorPlanCAD: recall", "BLD-AR: paredes", "BLD-AR: recall"],
           [[style, *style_cells(by_style[style].get("FloorPlanCAD")), *style_cells(by_style[style].get("BLD-AR"))]
            for style in ["vacía", "con trazos adentro", "relleno gris", "relleno negro", "sin interior"]
            if style in by_style])

## Wall por plano y escala

> Relación de espesor: espesor de muro del plano en el tile de 1024 px dividido por el de FloorPlanCAD (18 px).

In [ ]:
dominant = {}
for dataset, name, style, _, total in styles:
    if dataset == "BLD-AR":
        dominant.setdefault(name, Counter())[style] += total
rows = []
for plan in sorted(plans, key=lambda plan: plan["espesor"]):
    if plan["tp"] + plan["fn"] == 0:
        continue
    ratio = plan["espesor"] / FPC_WALL_THICKNESS
    rows.append([plan["plano"], f"{plan['espesor']:.1f}", ratio, dominant[plan["plano"]].most_common(1)[0][0],
                 *pixel_scores(plan["tp"], plan["fp"], plan["fn"])[2:]])
show_table(["Plano", "Espesor (px)", "Relación de espesor", "Estilo dominante", "Recall", "F1"], rows)

plt.figure(figsize=(7, 4))
plt.scatter([row[2] for row in rows], [row[5] for row in rows])
plt.xscale("log")
plt.axvline(1, color="gray", linestyle="--")
plt.xlabel("Espesor de muro / espesor en FloorPlanCAD")
plt.ylabel("F1 de wall por píxel")
plt.show()

## Tiles adaptados al grosor de muro

> El tile de cada plano mide 1024 × espesor de muro / 18 px, para que la red vea los muros con el grosor de FloorPlanCAD. Se compara contra el tile fijo de 1024 px con la misma unión de SAHI que `evaluate_segmentation_model.ipynb`, solo para wall.

In [ ]:
def coco_annotation(points, image_id, iscrowd):
    x, y = points[:, 0], points[:, 1]
    return {
        "image_id": image_id, "category_id": 1, "iscrowd": iscrowd, "segmentation": [points.ravel().tolist()],
        "bbox": [float(x.min()), float(y.min()), float(x.max() - x.min()), float(y.max() - y.min())],
        "area": float(0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))),
    }


images, annotations = [], []
for image_id, data in enumerate(bldar.values(), start=1):
    images.append({"id": image_id, "file_name": data["imagePath"], "width": data["imageWidth"],
                   "height": data["imageHeight"]})
    # Las clases excluidas son zonas a ignorar (crowd de COCO), como en la evaluación
    annotations += [coco_annotation(shape_points(shape), image_id, iscrowd=int(shape["label"] != "wall"))
                    for shape in data["shapes"] if shape["label"] == "wall" or shape["label"] not in MAPPED]
for annotation_id, annotation in enumerate(annotations, start=1):
    annotation["id"] = annotation_id
coco_gt = COCO()
coco_gt.dataset = {"images": images, "annotations": annotations, "categories": [{"id": 1, "name": "wall"}]}
coco_gt.createIndex()


def wall_ap(predictions, iou_type):
    """AP50 y AP50-95 de wall."""
    # Copias: COCOeval reemplaza los polígonos de las predicciones por RLE
    evaluator = COCOeval(coco_gt, coco_gt.loadRes([dict(p) for p in predictions]), iou_type)
    evaluator.evaluate()
    evaluator.accumulate()
    precision = evaluator.eval["precision"][:, :, 0, 0, -1]
    return float(precision[0][precision[0] > -1].mean()), float(precision[precision > -1].mean())


sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(WEIGHTS),
    confidence_threshold=CONF, device=DEVICE, image_size=TILE,
)
runs = {"Tile fijo": [], "Tile adaptado": []}
pixels = {"Tile fijo": np.zeros(3, np.int64), "Tile adaptado": np.zeros(3, np.int64)}
for image, plan in zip(images, plans):
    data = bldar[plan["plano"]]
    adapted = TILE if np.isnan(plan["espesor"]) else round(TILE * plan["espesor"] / FPC_WALL_THICKNESS)
    adapted = int(np.clip(adapted, 256, min(image["width"], image["height"])))
    scale = min(1.0, MAX_SIDE / max(image["width"], image["height"]))
    labels = bldar_labels(data, scale)
    gt_wall, valid = labels == WALL_LABEL, ~np.isin(labels, IGNORED)
    for run, tile in [("Tile fijo", TILE), ("Tile adaptado", adapted)]:
        result = get_sliced_prediction(
            str(BLDAR / image["file_name"]), sahi_model,
            slice_height=tile, slice_width=tile,
            overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
            perform_standard_pred=False, verbose=0, force_postprocess_type=True,
        )
        walls = [p for p in result.to_coco_predictions(image_id=image["id"]) if p["category_name"] == "wall"]
        for prediction in walls:
            prediction["category_id"] = 1
        runs[run] += walls
        pred_wall = union_mask([np.array(polygon).reshape(-1, 2) * scale for p in walls if p["score"] >= PIXEL_CONF
                                for polygon in p["segmentation"] if len(polygon) >= 6], labels.shape)
        pixels[run] += [(gt_wall & pred_wall).sum(), (~gt_wall & pred_wall & valid).sum(), (gt_wall & ~pred_wall).sum()]

gt_count = sum(not a["iscrowd"] for a in annotations)
rows = []
for run, predictions in runs.items():
    count = sum(p["score"] >= PIXEL_CONF for p in predictions)
    rows.append([run, *wall_ap(predictions, "bbox"), *wall_ap(predictions, "segm"), pixel_scores(*pixels[run])[3],
                 count])
show_table(["", "AP50 (B)", "AP50-95 (B)", "AP50 (S)", "AP50-95 (S)", "F1 por píxel",
            f"Paredes predichas (etiquetadas: {gt_count})"], rows)

## Matriz de confusión por píxel

> Con el tile fijo de 1024 px: qué predice el modelo sobre los píxeles etiquetados como wall en BLD-AR, y qué etiqueta tienen los píxeles que predice como wall.

In [ ]:
predicted_names = [yolo.names[i] for i in range(len(yolo.names))] + ["sin predicción"]
on_walls = confusion[WALL_LABEL] / confusion[WALL_LABEL].sum()
show_table(["Predicción sobre píxeles de wall", "Fracción"],
           [[predicted_names[i], float(on_walls[i])] for i in np.argsort(on_walls)[::-1][:8]])
as_walls = confusion[:, WALL] / max(confusion[:, WALL].sum(), 1)
show_table(["Etiqueta de BLD-AR de los píxeles predichos como wall", "Fracción"],
           [[LABELS[i], float(as_walls[i])] for i in np.argsort(as_walls)[::-1][:8]])

## Mapas de calor

> Por plano: etiquetas de wall en rojo, probabilidad de wall por píxel y mapa de activación (EigenCAM) sobre el plano.

In [ ]:
for name, (gray, walls, wall, cam) in heatmaps.items():
    fig, axes = plt.subplots(1, 3, figsize=(21, 7 * gray.shape[0] / gray.shape[1] + 1))
    axes[0].imshow(gray, cmap="gray")
    for points in walls:
        axes[0].fill(points[:, 0], points[:, 1], facecolor="none", edgecolor="red", linewidth=0.8)
    axes[1].imshow(wall, cmap="inferno", vmin=0, vmax=1)
    axes[2].imshow(gray, cmap="gray")
    axes[2].imshow(cam, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    for ax, title in zip(axes, ["Etiquetas de wall", "Probabilidad de wall", "Activación (EigenCAM)"]):
        ax.set_title(f"{name} · {title}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Límites de YOLO-seg con paredes

> Resolución de las máscaras: YOLO-seg arma cada máscara con prototipos a 1/4 de la entrada; el IoU máximo es el de las etiquetas de wall bajadas a esa resolución y vueltas a subir. Unión de SAHI: AP de wall si las predicciones fueran las etiquetas, sin unir y con GREEDYNMM (IOS 0.5). Tope de 300 detecciones por tile.

In [ ]:
def proto_bound(mask, factor):
    """Intersección y unión entre una máscara y la misma bajada `factor` veces y vuelta a subir."""
    h, w = mask.shape
    small = cv2.resize(mask.astype(np.float32), (max(1, round(w / factor)), max(1, round(h / factor))),
                       interpolation=cv2.INTER_AREA)
    restored = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5
    return np.array([(mask & restored).sum(), (mask | restored).sum()])


bounds = {"BLD-AR": np.zeros(2), "FloorPlanCAD": np.zeros(2)}
for data in bldar.values():
    scale = min(1.0, MAX_SIDE / max(data["imageWidth"], data["imageHeight"]))
    # Con tiles de 1024 px a imgsz 1024, un píxel del prototipo son 4 píxeles del plano
    bounds["BLD-AR"] += proto_bound(bldar_labels(data, scale) == WALL_LABEL, 4 * scale)
for path in fpc_images:
    W, H = Image.open(path).size
    bounds["FloorPlanCAD"] += proto_bound(union_mask(fpc_walls(path, W, H), (H, W)), 4 * max(H, W) / TILE)

merge = GreedyNMMPostprocess(match_threshold=0.5, match_metric="IOS", class_agnostic=False)
oracle = {"sin unir": [], "con GREEDYNMM": []}
for image, data in zip(images, bldar.values()):
    labelled = [ObjectPrediction(bbox=[*points.min(0), *points.max(0)], category_id=0, category_name="wall",
                                 segmentation=[points.ravel().tolist()], score=1.0,
                                 full_shape=[image["height"], image["width"]]) for points in bldar_walls(data)]
    if not labelled:
        continue
    for key, predictions in [("sin unir", labelled), ("con GREEDYNMM", merge(labelled))]:
        oracle[key] += [{**p.to_coco_prediction(image_id=image["id"]).json, "category_id": 1} for p in predictions]

detections = np.array(tiles)
show_table(["Límite", "Valor"], [
    ["IoU máximo de wall por la resolución de los prototipos, BLD-AR", float(bounds["BLD-AR"][0] / bounds["BLD-AR"][1])],
    ["IoU máximo de wall por la resolución de los prototipos, FloorPlanCAD",
     float(bounds["FloorPlanCAD"][0] / bounds["FloorPlanCAD"][1])],
    ["AP50 / AP50-95 (S) de wall con las etiquetas como predicciones, sin unir",
     " / ".join(f"{v:.3f}" for v in wall_ap(oracle["sin unir"], "segm"))],
    ["AP50 / AP50-95 (S) de wall con las etiquetas como predicciones, con GREEDYNMM",
     " / ".join(f"{v:.3f}" for v in wall_ap(oracle["con GREEDYNMM"], "segm"))],
    ["Paredes etiquetadas que GREEDYNMM une con otra", f"{len(oracle['sin unir']) - len(oracle['con GREEDYNMM'])} de {len(oracle['sin unir'])}"],
    ["Tiles de BLD-AR en el tope de 300 detecciones", f"{int((detections.sum(1) >= 300).sum())} de {len(detections)}"],
    ["Máximo de paredes detectadas en un tile", int(detections[:, WALL].max())],
])